# RAG Assignment

**Document:** Regulation (EU) 2016/679 (GDPR), 88 pages, 173 recitals and 99 articles.
I picked it because a rule and its exceptions sit far apart, it uses a lot of
negation, and the recitals restate the articles in looser wording.

| Stage | Component |
|---|---|
| Bi-encoder | `BAAI/bge-small-en-v1.5` (384-d, cosine via inner product) |
| Vector store | FAISS `IndexFlatIP` over normalised vectors (exact search) |
| Cross-encoder | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Generator | `qwen/qwen3.8-27b` via the Groq API, `temperature=0` |

**Pipeline 1:** top 10 by cosine similarity, the first 3 go to the LLM -> Answer 1.
**Pipeline 2:** the same 10 chunks re-scored by the cross-encoder, the top 3 go to the LLM -> Answer 2.

Prompt, model and settings are identical, so only the retrieved context differs.


## 1. Environment

In [1]:
# pypdf and groq are pinned. sentence-transformers and faiss only get minimum versions
# because Colab ships its own torch and hard pins can break the install. The versions
# actually used are recorded below.
%pip install -q "pypdf==6.20.0" "groq==0.33.0" "sentence-transformers>=5.1" "faiss-cpu>=1.13"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import time
import statistics
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore")

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)


def get_groq_key():
    """Colab Secrets -> Kaggle Secrets -> environment variable -> local api.txt."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        pass
    key = os.environ.get("GROQ_API_KEY")
    if key:
        return key
    # Local fallback: api.txt next to the notebook (gitignored). Needed because an
    # editor opened before the variable was set passes a stale environment to its kernel.
    for candidate in (Path("api.txt"), Path("../api.txt")):
        if candidate.exists():
            found = candidate.read_text(encoding="utf-8").strip()
            if found:
                return found
    return None


GROQ_KEY = get_groq_key()
assert GROQ_KEY, (
    "No API key found. Colab: key icon in the sidebar -> add GROQ_API_KEY. "
    "Kaggle: Add-ons -> Secrets. Local: set the GROQ_API_KEY environment variable, "
    "or drop the key into a file named api.txt beside this notebook."
)

BI_ENCODER = "BAAI/bge-small-en-v1.5"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "qwen/qwen3.8-27b"

# Encoders run on CUDA when a GPU is present and fall back to CPU otherwise.
# This changes wall-clock timings only: the ranks and the generated answers are
# identical either way, so the GPU is a convenience, never part of the argument.
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
except Exception:
    DEVICE, DEVICE_NAME = "cpu", "CPU"

TOP_K, TOP_N = 10, 3  # retrieve 10 candidates, pass 3 to the LLM


def resolved_versions():
    """Record what actually got installed, so the run is reproducible after the fact."""
    import importlib.metadata as md
    out = {}
    for pkg in ("pypdf", "groq", "sentence-transformers", "faiss-cpu", "torch",
                "transformers", "numpy"):
        try:
            out[pkg] = md.version(pkg)
        except Exception:
            out[pkg] = "not installed"
    return out


VERSIONS = resolved_versions()
print("environment ready")
for k, v in VERSIONS.items():
    print(f"  {k:22} {v}")
print(f"  {'device':22} {DEVICE} ({DEVICE_NAME})")

environment ready
  pypdf                  6.20.0
  groq                   0.33.0
  sentence-transformers  6.1.0
  faiss-cpu              1.15.1
  torch                  2.6.0+cu124
  transformers           5.20.1
  numpy                  2.2.6
  device                 cuda (NVIDIA GeForce RTX 4050 Laptop GPU)


## 2. Load the document

EUR-Lex often answers with an empty `HTTP 202` response instead of the PDF, so I
do not rely on downloading it. The notebook uses the copy in `data/` first and only
tries a download if that is missing. Every file is checked (size, `%PDF-` header,
88 pages) and deleted if it fails, so a broken file is never reused. If all
sources fail, upload the PDF by hand.


In [3]:
import hashlib
import urllib.request

from pypdf import PdfReader

PDF = Path("gdpr.pdf")

EXPECTED_PAGES = 88
EXPECTED_SHA256 = "bd84e63f5b622b739a83389afc3b30d240f792bb88d8eb03a816c9a82b0c2499"
MIN_BYTES = 500_000

# Optional but recommended: raw URL of the copy committed to your own repository,
# e.g. "https://raw.githubusercontent.com/<user>/<repo>/main/data/gdpr.pdf"
GDPR_PDF_URL = ""

LOCAL_CANDIDATES = [PDF, Path("data/gdpr.pdf"), Path("../data/gdpr.pdf")]
URL_CANDIDATES = [u for u in [
    GDPR_PDF_URL,
    "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
    "https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng/pdfa1a",
] if u]

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"),
    "Accept": "application/pdf,*/*",
}


def validate(data):
    """-> (ok, reason). Rejects 202-empty bodies, HTML error pages and truncation."""
    if len(data) < MIN_BYTES:
        return False, f"too small ({len(data):,} bytes; a 202/error page, not the PDF)"
    if data[:5] != b"%PDF-":
        return False, f"not a PDF (starts {data[:12]!r})"
    tmp = Path("_probe.pdf")
    try:
        tmp.write_bytes(data)
        pages = len(PdfReader(str(tmp)).pages)
    except Exception as exc:
        return False, f"unparseable ({type(exc).__name__})"
    finally:
        tmp.unlink(missing_ok=True)
    if pages != EXPECTED_PAGES:
        return False, f"wrong page count ({pages}, expected {EXPECTED_PAGES})"
    return True, f"{len(data):,} bytes, {pages} pages"


def acquire():
    # 1. a local copy that actually validates
    for path in LOCAL_CANDIDATES:
        if path.exists():
            ok, why = validate(path.read_bytes())
            if ok:
                if path != PDF:
                    PDF.write_bytes(path.read_bytes())
                print(f"using local copy {path} - {why}")
                return True
            print(f"local {path} rejected: {why}")
            path.unlink(missing_ok=True)   # never leave a poisoned file behind

    # 2. remote sources, each validated before being written
    for url in URL_CANDIDATES:
        label = url if len(url) < 76 else url[:73] + "..."
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=120) as resp:
                status, data = resp.status, resp.read()
            ok, why = validate(data)
            if ok:
                PDF.write_bytes(data)
                print(f"downloaded from {label}\n  {why}")
                return True
            print(f"rejected {label}\n  HTTP {status}: {why}")
        except Exception as exc:
            print(f"failed   {label}\n  {type(exc).__name__}: {exc}")
    return False


if not acquire():
    print("\n" + "=" * 70)
    print("Automatic download failed - EUR-Lex is refusing PDF requests.")
    print("Upload the file by hand; it is 0.98 MB and takes a few seconds.")
    print("  Get it from: https://eur-lex.europa.eu/eli/reg/2016/679/oj")
    print("  (open in a browser, save the English PDF as gdpr.pdf)")
    print("=" * 70)
    try:
        from google.colab import files        # Colab: file picker
        up = files.upload()
        name = next(iter(up))
        data = up[name]
        ok, why = validate(data)
        assert ok, f"uploaded file rejected: {why}"
        PDF.write_bytes(data)
        print(f"accepted upload {name} - {why}")
    except ImportError:                       # Kaggle / local
        raise SystemExit("Place a valid gdpr.pdf beside this notebook and re-run this cell.")

# final gate: nothing downstream runs on an unvalidated document
raw_bytes = PDF.read_bytes()
ok, why = validate(raw_bytes)
assert ok, f"gdpr.pdf is not usable: {why}"

digest = hashlib.sha256(raw_bytes).hexdigest()
reader = PdfReader(str(PDF))
print(f"\n{PDF.name}: {why}")
print(f"sha256: {digest}")
print("        matches the reference copy" if digest == EXPECTED_SHA256
      else "        NOTE: differs from the reference copy - EUR-Lex regenerates PDFs,\n"
           "        so this is expected if freshly downloaded. Page count is verified.")

using local copy gdpr.pdf - 982,296 bytes, 88 pages

gdpr.pdf: 982,296 bytes, 88 pages
sha256: bd84e63f5b622b739a83389afc3b30d240f792bb88d8eb03a816c9a82b0c2499
        matches the reference copy


## 3. Extract and clean

Problems I found in the raw text:

1. **Page headers.** Every page has the line
   `4.5.2016 L 119/n Official Journal of the European Union EN`. I remove it as a
   whole line, not as a phrase, because Art 45(8), 92 and 99(1) also contain
   "Official Journal of the European Union".
2. **Hard line breaks.** pypdf keeps the PDF's line breaks, so sentences arrive
   split across lines. I join them back together.
3. **Article headings.** "Article 9", its title and its body are separate lines and
   have to stay separate. Otherwise the 16 single-paragraph articles (10, 16, 19, 29,
   ...) merge into their title and drop out of the corpus. Art 29 is one of them and
   is the gold answer for Q11.


In [4]:
# A page-furniture line is the only place "L 119/n" and the OJ title co-occur.
FURNITURE = re.compile(
    r"^.*(?=.*L\s*119/\d+)(?=.*Official Journal of the European Union).*$", re.M
)
ART_HEAD = re.compile(r"^Article\s+(\d+)$")
NEW_PARA = re.compile(r"^(CHAPTER\s|SECTION\s|\(\d{1,3}\)\s|\d{1,2}\.\s|\([a-z]\)\s)")


def extract_text(pdf_path):
    return "\n".join((p.extract_text() or "") for p in PdfReader(str(pdf_path)).pages)


def clean_text(text):
    """Drop page furniture, unwrap hard line breaks, keep Article/title/body apart."""
    text = FURNITURE.sub("", text).replace("­", "")  # also drop soft hyphens
    text = re.sub(r"[ \t]+", " ", text)

    out, buf, expect_title = [], "", False

    def flush():
        nonlocal buf
        if buf.strip():
            out.append(buf.strip())
        buf = ""

    for line in (l.strip() for l in text.split("\n")):
        if not line:
            flush()
            continue
        if ART_HEAD.match(line):          # "Article 29" -> its own paragraph
            flush()
            out.append(line)
            expect_title = True
            continue
        if expect_title:                  # the line right after it is that article's title
            flush()
            out.append(line)
            expect_title = False
            continue
        if NEW_PARA.match(line):          # a numbered/lettered sub-paragraph starts fresh
            flush()
            buf = line
        elif buf.endswith("-"):           # a word split across a line break
            buf = buf[:-1] + line
        else:
            buf = (buf + " " + line).strip() if buf else line
    flush()
    return "\n".join(out)


raw = extract_text(PDF)
cleaned = clean_text(raw)
print(f"raw   : {len(raw):,} chars / {len(raw.splitlines()):,} lines")
print(f"clean : {len(cleaned):,} chars / {len(cleaned.splitlines()):,} paragraphs")

raw   : 360,504 chars / 3,804 lines
clean : 350,513 chars / 1,200 paragraphs


## 4. Chunking

**220 words per chunk, 40-word overlap.** This is small enough that a rule and its
exception do not land in the same chunk (e.g. Art 9(1) and Art 9(2)(h)). If they did,
the failure I want to test would not exist. The self-test in section 5 checks this.

**I kept the recitals.** They rephrase the articles in looser language and act as
about 200 distractors. Dropping them would make the baseline look better than it
would on a real corpus.

Footnotes restart their numbering on every page (`(1) OJ C 229, ...`), so a `(n)`
only counts as a recital or article if `n` is the next number in the sequence.


In [5]:
MAX_WORDS, OVERLAP_WORDS = 220, 40

REC_START = re.compile(r"^\((\d{1,3})\)\s+(.+)$")
# Footnotes cite the Official Journal or the legislative procedure; recitals do not.
FOOTNOTE = re.compile(
    r"^\(\d{1,3}\)\s+(OJ\b|Position of the European Parliament|Opinion of)"
    r"|\(OJ [LC] \d",
    re.I,
)
SKIP_UNIT = re.compile(r"^(CHAPTER|SECTION)\b", re.I)


def parse_units(cleaned_text):
    """-> [{type, number, title, paras}] for 173 recitals + 99 articles."""
    paras = [p.strip() for p in cleaned_text.split("\n") if p.strip()]
    units, cur, n_art, n_rec, want_title = [], None, 0, 0, False

    for para in paras:
        m_art, m_rec = ART_HEAD.match(para), REC_START.match(para)

        if m_art and int(m_art.group(1)) == n_art + 1:       # sequence guard
            n_art += 1
            cur = {"type": "article", "number": n_art, "title": None, "paras": []}
            units.append(cur)
            want_title = True
            continue

        if want_title and cur is not None:
            cur["title"] = para
            want_title = False
            continue

        if m_rec and not n_art:                              # still inside the recitals
            if FOOTNOTE.search(para) or int(m_rec.group(1)) != n_rec + 1:
                continue                                     # a footnote, not a recital
            n_rec += 1
            cur = {"type": "recital", "number": n_rec, "title": None,
                   "paras": [m_rec.group(2)]}
            units.append(cur)
            continue

        if SKIP_UNIT.match(para):
            continue
        if cur is not None:
            cur["paras"].append(para)

    return units


def _markers(text):
    """Sub-paragraph markers inside a chunk, e.g. ['2','h'] -> cite it as Art 9(2)(h)."""
    out = [m.group(1) for m in re.finditer(r"(?:^|(?<= ))(\d{1,2})\.\s", text)]
    out += [m.group(1) for m in re.finditer(r"\(([a-z])\)\s", text)]
    return sorted(set(out))


def chunk_units(units):
    chunks = []
    for unit in units:
        label = (f"Article {unit['number']}" if unit["type"] == "article"
                 else f"Recital {unit['number']}")
        body = " ".join(unit["paras"]).strip()

        if not body:                      # single-paragraph article: the title IS the body
            body = (unit["title"] or "").strip()
            if not body:
                continue

        head = f"[{label}" + (f" - {unit['title']}]" if unit["title"] else "]")
        words, start, part = body.split(), 0, 0

        while start < len(words):
            piece = words[start:start + MAX_WORDS]
            chunks.append({
                "chunk_id": len(chunks),
                "type": unit["type"],
                "number": unit["number"],
                "title": unit["title"],
                "label": label,
                "part": part,
                "n_words": len(piece),
                "markers": _markers(" ".join(piece)),
                # the label is prepended so the LLM can cite its source
                "text": head + " " + " ".join(piece),
            })
            part += 1
            if start + MAX_WORDS >= len(words):
                break
            start += MAX_WORDS - OVERLAP_WORDS
    return chunks


units = parse_units(cleaned)
chunks = chunk_units(units)
arts = [u for u in units if u["type"] == "article"]
recs = [u for u in units if u["type"] == "recital"]

print(f"units : {len(recs)} recitals + {len(arts)} articles")
print(f"chunks: {len(chunks)} "
      f"(recital {sum(c['type'] == 'recital' for c in chunks)}, "
      f"article {sum(c['type'] == 'article' for c in chunks)})")
print(f"words/chunk: min={min(c['n_words'] for c in chunks)} "
      f"max={max(c['n_words'] for c in chunks)} "
      f"mean={sum(c['n_words'] for c in chunks) / len(chunks):.0f}")

json.dump(chunks, open(OUT / "chunks.json", "w", encoding="utf-8"), ensure_ascii=False)

units : 173 recitals + 99 articles
chunks: 405 (recital 204, article 201)
words/chunk: min=22 max=220 mean=142


## 5. Self-test

Checks that the PDF was parsed correctly before the index is built. The last check
matters most: if Art 9(1) and Art 9(2)(h) were in the same chunk, the main failure
case would not exist.


In [6]:
failures = []


def check(name, condition, detail=""):
    tag = "PASS" if condition else "FAIL"
    if not condition:
        failures.append(name)
    print(f"  [{tag}] {name}" + (f" :: {detail}" if detail else ""))


def labels_containing(probe):
    return sorted({c["label"] for c in chunks if probe in c["text"]})


def ids_containing(probe):
    return {c["chunk_id"] for c in chunks if probe in c["text"]}


print("=== A. structure")
check("recitals 1..173 contiguous",
      [u["number"] for u in recs] == list(range(1, 174)), f"n={len(recs)}")
check("articles 1..99 contiguous",
      [u["number"] for u in arts] == list(range(1, 100)), f"n={len(arts)}")
check("every article has a title", all(a["title"] for a in arts))
missing = [a["number"] for a in arts
           if not any(c["type"] == "article" and c["number"] == a["number"] for c in chunks)]
check("every article produced at least one chunk", not missing, f"missing={missing}")
check("every recital produced at least one chunk",
      all(any(c["type"] == "recital" and c["number"] == r["number"] for c in chunks)
          for r in recs))
check(f"no chunk exceeds {MAX_WORDS} words",
      max(c["n_words"] for c in chunks) <= MAX_WORDS)
check("chunk_ids are 0..n-1",
      [c["chunk_id"] for c in chunks] == list(range(len(chunks))))
check("every chunk carries its label", all(c["text"].startswith("[") for c in chunks))

print("\n=== B. page furniture removed, real text preserved")
corpus = " ".join(c["text"] for c in chunks)
for probe in ["L 119/", "4.5.2016", "Official Journal of the European Union EN"]:
    check(f"furniture absent: {probe!r}", probe not in corpus, f"count={corpus.count(probe)}")
oj = [c for c in chunks if "Official Journal of the European Union" in c["text"]]
check("OJ title survives only in real articles",
      bool(oj) and all(c["type"] == "article" for c in oj),
      f"{[c['label'] for c in oj]}")
check("Art 99 enter-into-force clause intact",
      any("enter into force on the twentieth day" in c["text"] for c in chunks))

print("\n=== C. operative text sits in the correct unit")
spot = [
    ("Article 9", "sexual orientation shall be prohibited"),
    ("Article 9", "provision of health or social care or treatment"),
    ("Article 33", "not later than 72 hours after having become aware of it"),
    ("Article 2", "purely personal or household activity"),
    ("Article 5", "no longer than is necessary for the purposes"),
    ("Article 22", "decision based solely on automated processing"),
    ("Article 29", "shall not process those data except on instructions from the controller"),
    ("Article 83", "20 000 000"),
    ("Article 37", "shall designate a data protection officer"),
    ("Recital 4", "serve mankind"),
]
for want, probe in spot:
    found = labels_containing(probe)
    check(f"{want:11} <- {probe[:44]!r}", want in found, f"in {found[:3]}")

print("\n=== D. critical: the failure mode must be reachable")
prohibition = ids_containing("sexual orientation shall be prohibited")
exception = ids_containing("provision of health or social care or treatment")
check("Art 9(1) and Art 9(2)(h) are in DIFFERENT chunks",
      bool(prohibition and exception and not prohibition & exception),
      f"9(1)={sorted(prohibition)} 9(2)(h)={sorted(exception)}")

art33 = {c["chunk_id"] for c in chunks if c["label"] == "Article 33"}
rec85 = {c["chunk_id"] for c in chunks if c["label"] == "Recital 85"}
check("Recital 85 exists as a separate distractor for the 72-hour rule",
      bool(art33 and rec85 and not art33 & rec85),
      f"Art33={sorted(art33)} Rec85={sorted(rec85)}")

print("\n" + "=" * 64)
assert not failures, f"preprocessing self-test failed: {failures}"
print("SELF-TEST: all checks passed")

=== A. structure
  [PASS] recitals 1..173 contiguous :: n=173
  [PASS] articles 1..99 contiguous :: n=99
  [PASS] every article has a title
  [PASS] every article produced at least one chunk :: missing=[]
  [PASS] every recital produced at least one chunk
  [PASS] no chunk exceeds 220 words
  [PASS] chunk_ids are 0..n-1
  [PASS] every chunk carries its label

=== B. page furniture removed, real text preserved
  [PASS] furniture absent: 'L 119/' :: count=0
  [PASS] furniture absent: '4.5.2016' :: count=0
  [PASS] furniture absent: 'Official Journal of the European Union EN' :: count=0
  [PASS] OJ title survives only in real articles :: ['Article 45', 'Article 92', 'Article 99']
  [PASS] Art 99 enter-into-force clause intact

=== C. operative text sits in the correct unit
  [PASS] Article 9   <- 'sexual orientation shall be prohibited' :: in ['Article 9']
  [PASS] Article 9   <- 'provision of health or social care or treatm' :: in ['Article 9']
  [PASS] Article 33  <- 'not later than 72 

## 6. Index the chunks (bi-encoder)

`bge-small-en-v1.5` is a reasonably strong small model, so the baseline is not weak
on purpose. The query prefix that BGE expects is added to queries (not documents).
Embeddings are normalised and stored in FAISS `IndexFlatIP`, so inner product equals
cosine similarity and the search is exact.


In [7]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# BGE models expect this instruction on the query side only, never on documents.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
corpus_emb = bi_encoder.encode(
    [c["text"] for c in chunks],
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")
index_build_s = time.perf_counter() - t0

index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb)

print(f"embedded {len(chunks)} chunks -> {corpus_emb.shape} in {index_build_s:.1f}s")
print(f"FAISS index: {index.ntotal} vectors, dim {corpus_emb.shape[1]}")
assert index.ntotal == len(chunks)
# normalised vectors have unit norm, so inner product == cosine similarity
assert np.allclose(np.linalg.norm(corpus_emb, axis=1), 1.0, atol=1e-4)

Batches: 100%|██████████| 7/7 [00:01<00:00,  3.79it/s]

embedded 405 chunks -> (405, 384) in 1.9s
FAISS index: 405 vectors, dim 384


In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)
print(f"cross-encoder loaded: {CROSS_ENCODER}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6358.77it/s]


cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


## 7. The two pipelines

`retrieve_candidates` runs once per query and both pipelines use its output. This
follows the assignment (Pipeline 2 re-ranks the top 10 from the baseline) and means
Pipeline 2's cost is Pipeline 1's cost plus the re-ranking step.


In [9]:
def retrieve_candidates(query, k=TOP_K):
    """Stage 1 (shared): bi-encoder -> FAISS -> top-k candidates. Returns (cands, timings)."""
    t0 = time.perf_counter()
    q_emb = bi_encoder.encode([QUERY_PREFIX + query],
                              normalize_embeddings=True).astype("float32")
    t_embed = time.perf_counter() - t0

    t0 = time.perf_counter()
    scores, idxs = index.search(q_emb, k)
    t_search = time.perf_counter() - t0

    cands = []
    for rank, (ci, sc) in enumerate(zip(idxs[0], scores[0]), start=1):
        c = dict(chunks[int(ci)])
        c["dense_rank"] = rank
        c["dense_score"] = float(sc)
        cands.append(c)
    return cands, {"embed_s": t_embed, "search_s": t_search}


def rerank(query, cands, n=TOP_N):
    """Stage 2: cross-encoder scores every (query, chunk) pair jointly, then re-sorts."""
    pairs = [(query, c["text"]) for c in cands]
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict(pairs)
    t_rerank = time.perf_counter() - t0

    out = []
    for c, s in zip(cands, ce_scores):
        c = dict(c)
        c["ce_score"] = float(s)
        out.append(c)
    out.sort(key=lambda c: c["ce_score"], reverse=True)
    for rank, c in enumerate(out, start=1):
        c["ce_rank"] = rank
    return out, {"rerank_s": t_rerank}


# quick smoke test on the headline query
_c, _t = retrieve_candidates("Can a hospital process a patient's medical records to provide treatment?")
_r, _t2 = rerank("Can a hospital process a patient's medical records to provide treatment?", _c)
print("baseline top-3:", [(c["label"], c["dense_rank"]) for c in _c[:3]])
print("reranked top-3:", [(c["label"], c["dense_rank"], c["ce_rank"]) for c in _r[:3]])
print(f"timings: embed={_t['embed_s']*1000:.1f}ms search={_t['search_s']*1000:.1f}ms "
      f"rerank={_t2['rerank_s']*1000:.1f}ms")

baseline top-3: [('Recital 63', 1), ('Recital 69', 2), ('Article 9', 3)]
reranked top-3: [('Recital 63', 1, 1), ('Article 9', 3, 2), ('Recital 91', 5, 3)]
timings: embed=28.2ms search=0.2ms rerank=172.5ms


## 8. Generation

Same prompt, model and `temperature=0` for both answers. The system prompt tells the
model to answer only from the context, so a wrong answer points to a retrieval
problem and not to the LLM. Groq's free tier allows about 8,000 tokens a minute and
each call uses about 1,400, so calls are spaced out and retried on 429 errors.


In [10]:
from groq import Groq

client = Groq(api_key=GROQ_KEY)

SYSTEM_PROMPT = (
    "You are a compliance assistant answering questions about the GDPR. "
    "Answer ONLY from the provided context. Quote the Article or Recital number "
    "you relied on. If the context does not contain the answer, say so explicitly "
    "rather than guessing. Do not use any knowledge beyond the context."
)

USER_TEMPLATE = "Context:\n{context}\n\nQuestion: {question}"

MIN_CALL_INTERVAL_S = 11.0  # paces requests under the ~8k tokens/minute free-tier cap
_last_call_at = [0.0]


def build_prompt(question, selected):
    """Identical construction for both pipelines - only `selected` differs."""
    context = "\n\n".join(c["text"] for c in selected)
    return USER_TEMPLATE.format(context=context, question=question)


def generate(question, selected, max_retries=5):
    """Returns (answer_text, latency_s, usage_dict)."""
    wait = MIN_CALL_INTERVAL_S - (time.time() - _last_call_at[0])
    if wait > 0:
        time.sleep(wait)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, selected)},
    ]

    delay = 5.0
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                temperature=0,
                max_tokens=500,
            )
            latency = time.perf_counter() - t0
            _last_call_at[0] = time.time()
            return resp.choices[0].message.content.strip(), latency, dict(
                prompt_tokens=resp.usage.prompt_tokens,
                completion_tokens=resp.usage.completion_tokens,
            )
        except Exception as exc:
            if "429" not in str(exc) and "rate" not in str(exc).lower():
                raise
            print(f"  rate limited, backing off {delay:.0f}s (attempt {attempt + 1})")
            time.sleep(delay)
            delay *= 2
    raise RuntimeError("exhausted retries against the Groq rate limit")

## 9. Queries and gold labels

14 queries: 12 failure probes and 2 controls. I first ran a pool of 26 candidate
queries through the baseline (see `DIAGNOSTIC_query_mining.ipynb`) and picked these
to cover every possible outcome.

| baseline rank of gold chunk | meaning |
|---|---|
| **1-3** | baseline already works, re-ranking cannot help |
| **4-10** | re-ranking can rescue it |
| **>10** | not in the 10 candidates, so the cross-encoder never sees it |

Re-ranking only changes the order, not what was retrieved. Of the 26 candidates, 15
were already in the top 3, 6 were outside the top 10, and only 5 had room for the
cross-encoder to help. Those 6 are still findable: recall@3/10/20/50/100 over the
pool was 15/20/23/25/26 out of 26. So the limit is the window of 10, which the
assignment fixes.

### Which query covers which failure mode

| Assignment bullet | Query | Outcome |
|---|---|---|
| Keyword trap / distractor | **Q2** - Art 33 and Recitals 85/86 share the breach vocabulary | #5 -> #2, rescued |
| Exceptions & corner cases | **Q9** - Art 9(1) prohibits, Art 9(2)(h) permits, in different chunks | #3 -> #2 |
| Negation & constraints | **Q1** - Art 22(1) negative right; **Q11** - Art 29 role limit | #4 -> #1 rescued; #1 -> #1 |
| Other: chunk at rank 4-10 promoted | **Q3** - Art 26 joint controllers | #8 -> #1, rescued |

In the pool there were four pure-negation queries (Art 2, Art 7, Art 29, Art 48) and
three of them already ranked #1-#2 in the baseline. Negation alone did not break this
bi-encoder. Q1 fails because negation is combined with a vocabulary gap ("algorithm"
and "loan" never appear in the GDPR), which is why it is labelled
`negation / jargon gap`.

**Gold probes.** Each probe is a verbatim fragment of the operative text, chosen by
reading the regulation. A chunk counts as gold if it contains one, whether it is an
article or a recital. My first version required an article, which was wrong: on Q1
the cross-encoder promotes Recital 71, which says a person should not be subject to
"automatic refusal of an online credit application" without human intervention. That
does answer the question. An assertion checks which units each probe can match, so a
loose fragment cannot give a false hit (Q2's first probe also matched Art 35 and
three recitals).


In [11]:
QUERIES = [
    # ===== queries with headroom (gold at rank 4-10) =====
    # All five are kept, including Q4 and Q5 where re-ranking does not help.
    dict(qid="Q1", category="negation / jargon gap",
         query="Can a bank refuse my loan application using only an algorithm with no human involved?",
         gold_label="Article 22 / Recital 71", gold_units=["Article 22", "Recital 71"],
         gold_probes=["decision based solely on automated processing",
                      "automatic refusal of an online credit application"],
         answer_keys=["solely", "automated", "human intervention"],
         why="Negation AND vocabulary gap at once. Art 22(1) is a negative right - 'the right "
             "NOT to be subject to' - and 'algorithm' and 'loan' occur 0 times in the corpus."),
    dict(qid="Q2", category="keyword trap / distractor",
         query="When must we tell the affected individuals themselves about a breach?",
         gold_label="Article 34", gold_units=["Article 34"],
         gold_probes=["communicate the personal data breach to the data subject without undue delay"],
         answer_keys=["without undue delay", "communicate"],
         why="Art 33 (notify the regulator) and Recitals 85/86 share almost all of the breach "
             "vocabulary and outrank the article that actually answers this."),
    dict(qid="Q3", category="buried clause / role limit",
         query="If two companies decide together how data is used, who is responsible?",
         gold_label="Article 26", gold_units=["Article 26"],
         gold_probes=["jointly determine the purposes and means of processing"],
         answer_keys=["joint controller"],
         why="Asked in plain language; the regulation says 'joint controllers'. Gold is buried "
             "at #8 - the assignment's 'vital chunk at ranks 4-10' case, exactly."),
    dict(qid="Q4", category="constraint / buried clause",
         query="Should privacy be built into a product while it is being designed?",
         gold_label="Article 25", gold_units=["Article 25"],
         gold_probes=["designed to implement data-protection principles"],
         answer_keys=["by design", "by default"],
         why="Headroom case that the cross-encoder does NOT rescue: gold enters at #4 and is "
             "pushed to #7. Kept because excluding it would flatter the re-ranker."),
    dict(qid="Q5", category="role limit / contract",
         query="What has to be written in the contract with a company that handles data for us?",
         gold_label="Article 28", gold_units=["Article 28"],
         gold_probes=["processes the personal data only on documented instructions"],
         answer_keys=["documented instructions", "contract"],
         why="Second headroom case the cross-encoder does not rescue (#6 -> #7). Two of the "
             "five rank-4-10 queries fail; that ratio is the honest headline."),

    # ================= the regression: re-ranking actively loses ===============
    dict(qid="Q6", category="regression probe",
         query="How long are we allowed to keep customer records?",
         gold_label="Article 5", gold_units=["Article 5"],
         gold_probes=["no longer than is necessary for the purposes"],
         answer_keys=["no longer than is necessary", "storage limitation"],
         why="The baseline puts the storage-limitation principle at #1 and re-ranking demotes "
             "it to #4. This is what the trade-off costs."),

    # ================= stage-1 ceiling: gold never enters the top 10 ===========
    dict(qid="Q7", category="recall ceiling (>10)",
         query="What security measures protect against hackers stealing our database?",
         gold_label="Article 32", gold_units=["Article 32"],
         gold_probes=["pseudonymisation and encryption of personal data"],
         answer_keys=["pseudonymisation", "encryption"],
         why="'hackers' and 'database' are absent from the corpus; all 10 candidates come back "
             "as recitals and Art 32 sits at rank 18. Re-ranking fixes ordering, not recall."),
    dict(qid="Q8", category="recall ceiling (>10)",
         query="Must we obey a foreign court order demanding European personal data?",
         gold_label="Article 48", gold_units=["Article 48"],
         gold_probes=["may only be recognised or enforceable in any manner if based on an "
                      "international agreement"],
         answer_keys=["international agreement", "mutual legal assistance"],
         why="Second case outside the candidate window. One such query could be luck; two "
             "establish that the stage-1 ceiling is systematic, not anecdotal."),

    # ================= no headroom: the baseline already succeeds ==============
    dict(qid="Q9", category="exception / corner case",
         query="Can a hospital process a patient's medical records in order to provide treatment?",
         gold_label="Article 9", gold_units=["Article 9"],
         gold_probes=["provision of health or social care or treatment"],
         answer_keys=["health or social care", "treatment"],
         why="Art 9(1) flatly prohibits health data; the exception is in 9(2)(h). The baseline "
             "already reaches it at #3, so this documents the category without a rescue."),
    dict(qid="Q10", category="exception / corner case",
         query="If someone asks us to delete their data, when can we refuse?",
         gold_label="Article 17 / Recital 65", gold_units=["Article 17", "Recital 65"],
         gold_probes=["for exercising the right of freedom of expression and information"],
         answer_keys=["freedom of expression"],
         why="The erasure carve-outs. Baseline finds them unaided - evidence that an "
             "'exception' query is not automatically hard."),
    dict(qid="Q11", category="negation / role limit",
         query="May staff working for a processor use personal data without instructions from the controller?",
         gold_label="Article 29", gold_units=["Article 29"],
         gold_probes=["shall not process those data except on instructions from the controller"],
         answer_keys=["instructions from the controller", "shall not process"],
         why="The assignment's 'specific role limits' case with no synonym problem to confound "
             "it: query and Art 29 share vocabulary and only the polarity is hard."),
    dict(qid="Q12", category="negation / scope",
         query="In which situations does this regulation not apply at all?",
         gold_label="Article 2 / Recital 18", gold_units=["Article 2", "Recital 18"],
         gold_probes=["purely personal or household activity"],
         answer_keys=["purely personal", "household"],
         why="Pure negation, no jargon gap. Lands at #2 unaided - with Q11 and Q8 this is the "
             "evidence that negation ALONE did not break this bi-encoder."),

    # ================= controls =================================================
    dict(qid="C1", category="CONTROL",
         query="What is the maximum administrative fine for an infringement of this regulation?",
         gold_label="Article 83", gold_units=["Article 83"],
         gold_probes=["20 000 000"],
         answer_keys=["20 000 000", "20,000,000", "4 %", "4%"],
         why="Control: high lexical overlap, answer exactly where you would expect it."),
    dict(qid="C2", category="CONTROL",
         query="What is the definition of personal data?",
         gold_label="Article 4", gold_units=["Article 4"],
         gold_probes=["any information relating to an identified or identifiable natural person"],
         answer_keys=["identified or identifiable"],
         why="Control: a definition, stated once, in the words the query already uses."),
]


def gold_chunk_ids(q):
    """Gold = any chunk carrying the operative text, article OR recital."""
    return {c["chunk_id"] for c in chunks
            if any(p in c["text"] for p in q["gold_probes"])}


n_fail = sum(q["category"] != "CONTROL" for q in QUERIES)
n_ctrl = sum(q["category"] == "CONTROL" for q in QUERIES)
print(f"{len(QUERIES)} queries ({n_fail} failure probes, {n_ctrl} controls)\n")

for q in QUERIES:
    gid = gold_chunk_ids(q)
    assert gid, f"{q['qid']}: gold probes resolved to no chunk - fix the probe"
    units = sorted({chunks[c]["label"] for c in gid})
    # a loose probe must not be able to manufacture a hit on the wrong unit
    assert units == q["gold_units"], (
        f"{q['qid']}: probes resolve to {units}, expected {q['gold_units']}")
    print(f"  {q['qid']:<4} gold={q['gold_label']:24} chunks={sorted(gid)}")

14 queries (12 failure probes, 2 controls)

  Q1   gold=Article 22 / Recital 71  chunks=[84, 252]
  Q2   gold=Article 34               chunks=[275]
  Q3   gold=Article 26               chunks=[258]
  Q4   gold=Article 25               chunks=[256]
  Q5   gold=Article 28               chunks=[261, 262]
  Q6   gold=Article 5                chunks=[216, 217]
  Q7   gold=Article 32               chunks=[271]
  Q8   gold=Article 48               chunks=[315]
  Q9   gold=Article 9                chunks=[226]
  Q10  gold=Article 17 / Recital 65  chunks=[76, 77, 244]
  Q11  gold=Article 29               chunks=[266]
  Q12  gold=Article 2 / Recital 18   chunks=[19, 205]
  C1   gold=Article 83               chunks=[384, 385]
  C2   gold=Article 4                chunks=[208]


## 10. Run both pipelines

In [12]:
def cites_gold(answer, q):
    """Heuristic: does the answer cite the unit the operative text actually lives in?"""
    return any(u.lower() in answer.lower() for u in q["gold_units"])


def has_key_fact(answer, q):
    """Heuristic: does the answer contain the fact that makes it correct?"""
    a = answer.lower()
    return any(k.lower() in a for k in q["answer_keys"])


def rank_of_gold(ordered, gold_ids, key):
    """1-indexed position of the first gold chunk in `ordered`, or None if absent."""
    for c in ordered:
        if c["chunk_id"] in gold_ids:
            return c[key]
    return None


records = []
for q in QUERIES:
    gold_ids = gold_chunk_ids(q)
    print(f"\n=== {q['qid']} [{q['category']}]  {q['query']}")

    cands, t_ret = retrieve_candidates(q["query"])
    reranked, t_rr = rerank(q["query"], cands)

    base_top3 = cands[:TOP_N]                     # dense order, untouched
    rr_top3 = reranked[:TOP_N]                    # cross-encoder order

    ans1, lat1, use1 = generate(q["query"], base_top3)
    ans2, lat2, use2 = generate(q["query"], rr_top3)

    rec = dict(
        qid=q["qid"], category=q["category"], query=q["query"],
        gold_article=q["gold_label"], gold_probes=q["gold_probes"],
        why=q["why"], gold_chunk_ids=sorted(gold_ids),
        candidates=[{k: c[k] for k in
                     ("chunk_id", "label", "part", "dense_rank", "dense_score",
                      "ce_rank", "ce_score")} for c in reranked],
        baseline_top3=[c["label"] for c in base_top3],
        reranked_top3=[c["label"] for c in rr_top3],
        gold_rank_baseline=rank_of_gold(cands, gold_ids, "dense_rank"),
        gold_rank_reranked=rank_of_gold(reranked, gold_ids, "ce_rank"),
        gold_in_top3_baseline=any(c["chunk_id"] in gold_ids for c in base_top3),
        gold_in_top3_reranked=any(c["chunk_id"] in gold_ids for c in rr_top3),
        answer_1=ans1, answer_2=ans2,
        ans1_cites_gold=cites_gold(ans1, q), ans2_cites_gold=cites_gold(ans2, q),
        ans1_has_key_fact=has_key_fact(ans1, q), ans2_has_key_fact=has_key_fact(ans2, q),
        timings={**t_ret, **t_rr, "llm_1_s": lat1, "llm_2_s": lat2},
        usage={"answer_1": use1, "answer_2": use2},
    )
    records.append(rec)

    print(f"  baseline top-3: {rec['baseline_top3']}")
    print(f"  reranked top-3: {rec['reranked_top3']}")
    print(f"  gold rank: #{rec['gold_rank_baseline']} -> #{rec['gold_rank_reranked']}   "
          f"in top-3: {rec['gold_in_top3_baseline']} -> {rec['gold_in_top3_reranked']}")

json.dump(records, open(OUT / "results.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(f"\nsaved {len(records)} records -> {OUT / 'results.json'}")


=== Q1 [negation / jargon gap]  Can a bank refuse my loan application using only an algorithm with no human involved?
  baseline top-3: ['Recital 18', 'Recital 43', 'Recital 28']
  reranked top-3: ['Recital 71', 'Recital 18', 'Recital 80']
  gold rank: #4 -> #1   in top-3: False -> True

=== Q2 [keyword trap / distractor]  When must we tell the affected individuals themselves about a breach?
  baseline top-3: ['Recital 86', 'Recital 85', 'Article 33']
  reranked top-3: ['Recital 85', 'Article 34', 'Recital 86']
  gold rank: #5 -> #2   in top-3: False -> True

=== Q3 [buried clause / role limit]  If two companies decide together how data is used, who is responsible?
  baseline top-3: ['Recital 40', 'Recital 74', 'Recital 79']
  reranked top-3: ['Article 26', 'Recital 79', 'Article 47']
  gold rank: #8 -> #1   in top-3: False -> True

=== Q4 [constraint / buried clause]  Should privacy be built into a product while it is being designed?
  baseline top-3: ['Recital 78', 'Recital 63', 'Re

## 11. Rank-shift analysis

| metric | what it shows |
|---|---|
| **Recall@3** | did the operative text reach the LLM's context? |
| **Recall@10** | did stage 1 find it at all? Same for both pipelines, so it caps what the cross-encoder can do |
| **MRR** | how far a chunk moved (Recall@3 treats #8 -> #1 and #4 -> #3 the same) |
| **answer checks** | two keyword tests on the generated text |

All 14 queries are counted. If the gold chunk is outside the top 10 it scores 0
instead of being dropped.

**Limitation.** `cites_gold` matches the unit label ("Article 28"), so any chunk of
that article satisfies it. It is a rough proxy, not a correctness test.
`has_key_fact` is a plain substring test. Section 12 prints both answers so they can
be read directly.


In [13]:
def fmt_rank(r):
    return f"#{r}" if r else "not in top-10"


print(f"{'Q':<4}{'category':<28}{'gold':<24}{'baseline':>15}{'reranked':>15}  effect")
print("-" * 100)
for r in records:
    b, a = r["gold_rank_baseline"], r["gold_rank_reranked"]
    if r["gold_in_top3_baseline"] and r["gold_in_top3_reranked"]:
        effect = "both OK"
    elif not r["gold_in_top3_baseline"] and r["gold_in_top3_reranked"]:
        effect = "RESCUED by re-ranking"
    elif r["gold_in_top3_baseline"] and not r["gold_in_top3_reranked"]:
        effect = "REGRESSED by re-ranking"
    else:
        effect = "both failed"
    print(f"{r['qid']:<4}{r['category']:<28}{r['gold_article']:<24}"
          f"{fmt_rank(b):>15}{fmt_rank(a):>15}  {effect}")

Q   category                    gold                           baseline       reranked  effect
----------------------------------------------------------------------------------------------------
Q1  negation / jargon gap       Article 22 / Recital 71              #4             #1  RESCUED by re-ranking
Q2  keyword trap / distractor   Article 34                           #5             #2  RESCUED by re-ranking
Q3  buried clause / role limit  Article 26                           #8             #1  RESCUED by re-ranking
Q4  constraint / buried clause  Article 25                           #4             #7  both failed
Q5  role limit / contract       Article 28                           #6             #7  both failed
Q6  regression probe            Article 5                            #1             #4  REGRESSED by re-ranking
Q7  recall ceiling (>10)        Article 32                not in top-10  not in top-10  both failed
Q8  recall ceiling (>10)        Article 48                not 

In [14]:
# Metrics: Recall@3 (reached the context?), Recall@10 (found at all? the ceiling),
# MRR (how far it moved) and two keyword checks on the answers. All queries are
# counted; a gold chunk outside the top 10 scores 0 and is not dropped.

n = len(records)
base_hits = sum(r["gold_in_top3_baseline"] for r in records)
rr_hits = sum(r["gold_in_top3_reranked"] for r in records)

fails = [r for r in records if r["category"] != "CONTROL"]
ctrls = [r for r in records if r["category"] == "CONTROL"]


def rr(rank):
    """Reciprocal rank; 0 when the chunk was never retrieved."""
    return 1.0 / rank if rank else 0.0


def pct(x, total):
    return f"{x}/{total} ({x / total:.0%})"


recall10 = sum(r["gold_rank_baseline"] is not None for r in records)
mrr_base = sum(rr(r["gold_rank_baseline"]) for r in records) / n
mrr_rr = sum(rr(r["gold_rank_reranked"]) for r in records) / n

print(f"{'metric':<34}{'baseline':>14}{'re-ranked':>14}   delta")
print("-" * 80)
print(f"{'Recall@3  (all)':<34}{pct(base_hits, n):>14}{pct(rr_hits, n):>14}"
      f"   {rr_hits - base_hits:+d}")
print(f"{'Recall@3  (failure probes)':<34}"
      f"{pct(sum(r['gold_in_top3_baseline'] for r in fails), len(fails)):>14}"
      f"{pct(sum(r['gold_in_top3_reranked'] for r in fails), len(fails)):>14}"
      f"   {sum(r['gold_in_top3_reranked'] for r in fails) - sum(r['gold_in_top3_baseline'] for r in fails):+d}")
print(f"{'Recall@3  (controls)':<34}"
      f"{pct(sum(r['gold_in_top3_baseline'] for r in ctrls), len(ctrls)):>14}"
      f"{pct(sum(r['gold_in_top3_reranked'] for r in ctrls), len(ctrls)):>14}"
      f"   {sum(r['gold_in_top3_reranked'] for r in ctrls) - sum(r['gold_in_top3_baseline'] for r in ctrls):+d}")
print(f"{'Recall@10 (stage-1 ceiling)':<34}{pct(recall10, n):>14}{pct(recall10, n):>14}"
      f"   shared")
print(f"{'MRR':<34}{mrr_base:>14.3f}{mrr_rr:>14.3f}   {mrr_rr - mrr_base:+.3f}")
print("-" * 80)

a1_cite = sum(r["ans1_cites_gold"] for r in records)
a2_cite = sum(r["ans2_cites_gold"] for r in records)
a1_key = sum(r["ans1_has_key_fact"] for r in records)
a2_key = sum(r["ans2_has_key_fact"] for r in records)
print(f"{'Answer cites the gold unit':<34}{pct(a1_cite, n):>14}{pct(a2_cite, n):>14}"
      f"   {a2_cite - a1_cite:+d}")
print(f"{'Answer contains the key fact':<34}{pct(a1_key, n):>14}{pct(a2_key, n):>14}"
      f"   {a2_key - a1_key:+d}")
print("\n(The two answer rows are keyword heuristics, not a judgement of quality.")
print(" For the report, read Answer 1 against Answer 2 in outputs/answers.md and")
print(" say what actually went wrong - omitted clause, wrong authority, or guess.)")

# ---------------------------------------------------------------- by category
print("\n\nBy failure mode")
print("-" * 80)
print(f"{'category':<30}{'n':>3}{'R@3 base':>11}{'R@3 rr':>9}{'rescued':>9}{'regressed':>11}")
print("-" * 80)
for cat in dict.fromkeys(r["category"] for r in records):
    rows = [r for r in records if r["category"] == cat]
    b = sum(r["gold_in_top3_baseline"] for r in rows)
    a = sum(r["gold_in_top3_reranked"] for r in rows)
    res = sum(not r["gold_in_top3_baseline"] and r["gold_in_top3_reranked"] for r in rows)
    reg = sum(r["gold_in_top3_baseline"] and not r["gold_in_top3_reranked"] for r in rows)
    print(f"{cat:<30}{len(rows):>3}{b:>11}{a:>9}{res:>9}{reg:>11}")

# ------------------------------------------------- where the movement came from
head = [r for r in records if r["gold_rank_baseline"] and r["gold_rank_baseline"] > 3]
head_res = sum(r["gold_in_top3_reranked"] for r in head)
print("\n" + "-" * 80)
print(f"Queries with headroom (gold at rank 4-10): {len(head)}")
print(f"  rescued into the top 3 : {head_res}/{len(head)}  -> {[r['qid'] for r in head if r['gold_in_top3_reranked']]}")
print(f"  left outside           : {len(head) - head_res}/{len(head)}  -> {[r['qid'] for r in head if not r['gold_in_top3_reranked']]}")
print(f"Queries with no headroom (gold already #1-#3): "
      f"{sum(1 for r in records if r['gold_rank_baseline'] and r['gold_rank_baseline'] <= 3)}")
print(f"Queries outside the window (gold not in top 10): "
      f"{sum(1 for r in records if r['gold_rank_baseline'] is None)}")


metric                                  baseline     re-ranked   delta
--------------------------------------------------------------------------------
Recall@3  (all)                       7/14 (50%)    9/14 (64%)   +2
Recall@3  (failure probes)            5/12 (42%)    7/12 (58%)   +2
Recall@3  (controls)                  2/2 (100%)    2/2 (100%)   +0
Recall@10 (stage-1 ceiling)          12/14 (86%)   12/14 (86%)   shared
MRR                                        0.487         0.574   +0.086
--------------------------------------------------------------------------------
Answer cites the gold unit            7/14 (50%)   10/14 (71%)   +3
Answer contains the key fact         13/14 (93%)   12/14 (86%)   -1

(The two answer rows are keyword heuristics, not a judgement of quality.
 For the report, read Answer 1 against Answer 2 in outputs/answers.md and
 say what actually went wrong - omitted clause, wrong authority, or guess.)


By failure mode
-----------------------------------------

## 12. Answer comparison

Both answers for every query, printed in full.


In [15]:
for r in records:
    changed = r["gold_in_top3_baseline"] != r["gold_in_top3_reranked"]
    flag = "  <-- context changed" if changed else ""
    print("=" * 100)
    print(f"{r['qid']} [{r['category']}]{flag}")
    print(f"Q: {r['query']}")
    print(f"gold: {r['gold_article']}   rank {fmt_rank(r['gold_rank_baseline'])} -> "
          f"{fmt_rank(r['gold_rank_reranked'])}")
    print(f"why : {r['why']}")
    print(f"\n--- ANSWER 1 (baseline context: {r['baseline_top3']})")
    print(r["answer_1"])
    print(f"\n--- ANSWER 2 (reranked context: {r['reranked_top3']})")
    print(r["answer_2"])
    print()

Q1 [negation / jargon gap]  <-- context changed
Q: Can a bank refuse my loan application using only an algorithm with no human involved?
gold: Article 22 / Recital 71   rank #4 -> #1
why : Negation AND vocabulary gap at once. Art 22(1) is a negative right - 'the right NOT to be subject to' - and 'algorithm' and 'loan' occur 0 times in the corpus.

--- ANSWER 1 (baseline context: ['Recital 18', 'Recital 43', 'Recital 28'])
The provided context does not contain the answer to this question. The excerpts provided (Recitals 18, 43, and 28) discuss the scope of the Regulation regarding household activities, the conditions for valid consent, and the use of pseudonymization, but they do not address automated decision-making, algorithmic lending, or the right to human intervention in credit decisions.

--- ANSWER 2 (reranked context: ['Recital 71', 'Recital 18', 'Recital 80'])
Based on the provided context, a bank generally **cannot** refuse a loan application using only an algorithm with no hu

## 13. Latency profiling

Median of 10 runs after discarding a warm-up run, with each stage timed separately.
The LLM call is timed on its own because network variance on the Groq API is much
larger than the cross-encoder's cost and would hide it. Model loading time is not
included.


In [16]:
N_RUNS = 10
probe_query = QUERIES[0]["query"]

# warm-up, discarded: first call pays lazy init and cache costs
retrieve_candidates(probe_query)
rerank(probe_query, retrieve_candidates(probe_query)[0])

embed_ms, search_ms, rerank_ms = [], [], []
for _ in range(N_RUNS):
    cands, t_ret = retrieve_candidates(probe_query)
    _, t_rr = rerank(probe_query, cands)
    embed_ms.append(t_ret["embed_s"] * 1000)
    search_ms.append(t_ret["search_s"] * 1000)
    rerank_ms.append(t_rr["rerank_s"] * 1000)

med = statistics.median
q_embed, q_search, q_rerank = med(embed_ms), med(search_ms), med(rerank_ms)
p1_retrieval = q_embed + q_search
p2_retrieval = p1_retrieval + q_rerank

llm1 = med([r["timings"]["llm_1_s"] * 1000 for r in records])
llm2 = med([r["timings"]["llm_2_s"] * 1000 for r in records])

print(f"median of {N_RUNS} runs, warm-up discarded, {len(chunks)} chunks indexed\n")
print(f"{'stage':<34}{'median (ms)':>14}")
print("-" * 48)
print(f"{'query embedding (bi-encoder)':<34}{q_embed:>14.1f}")
print(f"{'FAISS search (top-10)':<34}{q_search:>14.1f}")
print(f"{'cross-encoder rerank (10 pairs)':<34}{q_rerank:>14.1f}")
print("-" * 48)
print(f"{'PIPELINE 1 retrieval total':<34}{p1_retrieval:>14.1f}")
print(f"{'PIPELINE 2 retrieval total':<34}{p2_retrieval:>14.1f}")
print(f"{'  -> re-ranking overhead':<34}{q_rerank:>14.1f}  "
      f"({q_rerank / p1_retrieval:.1f}x baseline retrieval)")
print("-" * 48)
print(f"{'LLM generation, Answer 1':<34}{llm1:>14.1f}")
print(f"{'LLM generation, Answer 2':<34}{llm2:>14.1f}")
print(f"\nper-candidate cross-encoder cost: {q_rerank / TOP_K:.2f} ms")
print(f"one-off index build ({len(chunks)} chunks): {index_build_s:.1f}s")

median of 10 runs, warm-up discarded, 405 chunks indexed

stage                                median (ms)
------------------------------------------------
query embedding (bi-encoder)                11.6
FAISS search (top-10)                        0.1
cross-encoder rerank (10 pairs)             23.8
------------------------------------------------
PIPELINE 1 retrieval total                  11.7
PIPELINE 2 retrieval total                  35.4
  -> re-ranking overhead                    23.8  (2.0x baseline retrieval)
------------------------------------------------
LLM generation, Answer 1                   452.6
LLM generation, Answer 2                   640.7

per-candidate cross-encoder cost: 2.38 ms
one-off index build (405 chunks): 1.9s


## 14. Why not cross-encode the whole corpus?

Measured directly: score the query against every chunk and compare with scoring
only the 10 candidates.


In [17]:
all_pairs = [(probe_query, c["text"]) for c in chunks]

t0 = time.perf_counter()
_ = cross_encoder.predict(all_pairs)
full_s = time.perf_counter() - t0

per_pair_ms = full_s * 1000 / len(chunks)
print(f"cross-encoding all {len(chunks)} chunks : {full_s:.2f}s "
      f"({per_pair_ms:.2f} ms/pair)")
print(f"cross-encoding {TOP_K} candidates      : {q_rerank / 1000:.3f}s")
print(f"ratio                              : {full_s / (q_rerank / 1000):.0f}x slower")
print()
print(f"The bi-encoder embeds the corpus once ({index_build_s:.1f}s) and then answers")
print(f"each query in {p1_retrieval:.1f}ms, because a document's embedding is computed")
print("independently of the query and can therefore be precomputed and indexed.")
print("A cross-encoder puts query and document through joint self-attention, so its")
print("score exists only for a specific pair - there is no per-document vector to")
print(f"cache. Every query costs N forward passes: {per_pair_ms:.2f}ms x N.")
print()
for scale in (len(chunks), 10_000, 1_000_000):
    print(f"  {scale:>9,} chunks -> {per_pair_ms * scale / 1000:>10.1f}s per query")
print()
print("Hence retrieve-then-rerank: the bi-encoder cuts N to a constant 10, and the")
print("cross-encoder pays its quadratic-attention cost only on that shortlist.")

cross-encoding all 405 chunks : 0.63s (1.56 ms/pair)
cross-encoding 10 candidates      : 0.024s
ratio                              : 27x slower

The bi-encoder embeds the corpus once (1.9s) and then answers
each query in 11.7ms, because a document's embedding is computed
independently of the query and can therefore be precomputed and indexed.
A cross-encoder puts query and document through joint self-attention, so its
score exists only for a specific pair - there is no per-document vector to
cache. Every query costs N forward passes: 1.56ms x N.

        405 chunks ->        0.6s per query
     10,000 chunks ->       15.6s per query
  1,000,000 chunks ->     1564.5s per query

Hence retrieve-then-rerank: the bi-encoder cuts N to a constant 10, and the
cross-encoder pays its quadratic-attention cost only on that shortlist.


## 15. Save outputs

Written to `outputs/` so the numbers can be checked without re-running the notebook
or needing an API key.


In [18]:
summary = dict(
    versions=VERSIONS,
    corpus=dict(document="Regulation (EU) 2016/679 (GDPR)", pages=88,
                recitals=len(recs), articles=len(arts), chunks=len(chunks),
                max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS),
    models=dict(bi_encoder=BI_ENCODER, cross_encoder=CROSS_ENCODER, llm=LLM_MODEL,
                top_k=TOP_K, top_n=TOP_N),
    recall_at_3=dict(
        overall=dict(baseline=base_hits, reranked=rr_hits, n=n),
        failure_probes=dict(
            baseline=sum(r["gold_in_top3_baseline"] for r in fails),
            reranked=sum(r["gold_in_top3_reranked"] for r in fails), n=len(fails)),
        controls=dict(
            baseline=sum(r["gold_in_top3_baseline"] for r in ctrls),
            reranked=sum(r["gold_in_top3_reranked"] for r in ctrls), n=len(ctrls)),
    ),
    # every metric over ALL queries: a gold chunk that never entered the top 10
    # counts as a miss (RR 0) rather than being dropped from the denominator
    metrics=dict(
        n_queries=n,
        recall_at_10_stage1=recall10,
        mrr=dict(baseline=round(mrr_base, 4), reranked=round(mrr_rr, 4)),
        answer_cites_gold_unit=dict(baseline=a1_cite, reranked=a2_cite, n=n),
        answer_has_key_fact=dict(baseline=a1_key, reranked=a2_key, n=n),
        headroom=dict(
            n_rank_4_to_10=len(head), rescued=head_res,
            no_headroom=sum(1 for r in records
                            if r["gold_rank_baseline"] and r["gold_rank_baseline"] <= 3),
            outside_window=sum(1 for r in records if r["gold_rank_baseline"] is None)),
    ),
    latency_ms=dict(query_embed=q_embed, faiss_search=q_search,
                    rerank_10_pairs=q_rerank, pipeline1_retrieval=p1_retrieval,
                    pipeline2_retrieval=p2_retrieval, llm_answer_1=llm1,
                    llm_answer_2=llm2, per_candidate_rerank=q_rerank / TOP_K,
                    n_runs=N_RUNS),
    scaling=dict(full_corpus_rerank_s=full_s, per_pair_ms=per_pair_ms,
                 index_build_s=index_build_s),
    rank_shifts=[dict(qid=r["qid"], category=r["category"],
                      gold_article=r["gold_article"],
                      baseline_rank=r["gold_rank_baseline"],
                      reranked_rank=r["gold_rank_reranked"],
                      baseline_top3=r["baseline_top3"],
                      reranked_top3=r["reranked_top3"]) for r in records],
)

json.dump(summary, open(OUT / "summary.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

with open(OUT / "answers.md", "w", encoding="utf-8") as fh:
    fh.write("# Answer 1 (baseline) vs Answer 2 (re-ranked)\n\n")
    for r in records:
        fh.write(f"## {r['qid']} - {r['category']}\n\n")
        fh.write(f"**Query:** {r['query']}\n\n")
        fh.write(f"**Gold:** {r['gold_article']} - rank "
                 f"{fmt_rank(r['gold_rank_baseline'])} -> "
                 f"{fmt_rank(r['gold_rank_reranked'])}\n\n")
        fh.write(f"**Baseline context:** {r['baseline_top3']}\n\n")
        fh.write(f"**Answer 1:**\n\n> {r['answer_1']}\n\n")
        fh.write(f"**Re-ranked context:** {r['reranked_top3']}\n\n")
        fh.write(f"**Answer 2:**\n\n> {r['answer_2']}\n\n---\n\n")

for p in sorted(OUT.iterdir()):
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

  outputs\answers.md  (24,523 bytes)
  outputs\chunks.json  (448,931 bytes)
  outputs\results.json  (72,002 bytes)
  outputs\summary.json  (7,254 bytes)
